In [0]:
-- ╔════════════════════════════════════════════════════════════╗
-- ║  01_Business_Silver_Cleaned                               ║
-- ║  Silver_Cleaned Business Tables — Empty DDL Only          ║
-- ╚════════════════════════════════════════════════════════════╝
--
-- Source : yelp_poc_uc.bronze_final.yelp_business (150,346 rows)
-- Targets (all in yelp_poc_uc.silver_cleaned):
--   yelp_business                       (parent)
--   yelp_business_attributes            (child, 1:1)  — type-corrected
--   yelp_business_hours                 (child, 1:1)  — type-corrected
--   yelp_business_categories            (child, 1:N)  — exploded rows
--   yelp_business_ambience              (child, 0:1)  — parsed dict
--   yelp_business_best_nights           (child, 0:1)  — parsed dict
--   yelp_business_parking               (child, 0:1)  — parsed dict
--   yelp_business_dietary_restrictions  (child, 0:1)  — parsed dict
--   yelp_business_good_for_meal         (child, 0:1)  — parsed dict
--   yelp_business_hair_specializes_in   (child, 0:1)  — parsed dict
--   yelp_business_music                 (child, 0:1)  — parsed dict
--
-- ════════════════════════════════════════════════════════════
-- RULES & DESIGN PRINCIPLES
-- ════════════════════════════════════════════════════════════
--
-- R1  DDL-Only / No Population
--     Tables are created empty. Data population is handled by
--     a separate loading script. This keeps DDL idempotent and
--     safe to re-run without side effects on existing data.
--
-- R2  No Unnecessary Constraint Removal
--     All constraints (PK, FK, NOT NULL) are declared inline
--     inside the CREATE TABLE statement. We never DROP a
--     constraint and re-add it unless there is a clear reason.
--     CREATE OR REPLACE TABLE resets the whole definition
--     atomically, so separate ALTER TABLE is unnecessary.
--
-- R3  Normalization of Nested Data
--     Nested or embedded source structures (STRUCTs, JSON-dict
--     strings, CSV-packed lists) are NOT flattened into the
--     parent table. Each nested domain becomes its own child
--     table. This applies at MULTIPLE nesting levels:
--       • Level 1 (Silver Cleaned): top-level STRUCTs and CSV
--         columns → child tables (e.g. attributes, hours,
--         categories)
--       • Level 2 (parsed dicts): dict-strings nested INSIDE
--         a STRUCT field → further child tables (e.g. Ambience,
--         Music, Parking are dicts inside attributes STRUCT)
--     Benefits:
--       • Cleaner schema and narrower reads
--       • Independent row-count semantics (e.g. many categories
--         per business)
--       • Easier evolution (adding attributes without widening
--         the parent)
--
-- R4  Referential Integrity via PK/FK
--     Every child table declares:
--       • A PRIMARY KEY on its business_id (or composite key)
--       • A FOREIGN KEY referencing the parent yelp_business(business_id)
--     These are informational in Databricks but document the
--     intended relationship and assist the query optimizer.
--
-- R5  Semantic Data Types
--     Column types reflect the real-world meaning of the data,
--     NOT the raw storage type from the bronze source. Type
--     mappings were inferred by inspecting actual bronze values:
--
--     Bronze Type       →  Silver Semantic Type  → Columns Affected
--     ────────────────────────────────────────────────────────────
--     BIGINT 0/1        →  BOOLEAN               → is_open
--     DOUBLE            →  DECIMAL(2,1)           → stars
--     STRING True/False →  BOOLEAN               → 24 attribute columns
--     STRING "1"-"4"    →  INT                   → attr_restaurants_price_range_2
--     STRING u'enum'    →  STRING (cleaned enum) → 7 categorical attributes
--     STRING {dict}     →  BOOLEAN (parsed)      → 49 flags in 7 parsed-dict tables
--     STRING HH:MM-HH:MM→ STRING (time range)   → 7 hours columns
--     STRING CSV list   →  Exploded rows         → categories child table
--
-- R6  No Ingestion Timestamps
--     No _ingestion_timestamp columns are added. Lineage and
--     provenance are tracked via TBLPROPERTIES metadata:
--       • pipelines.sourceTable → identifies the upstream bronze table
--       • design.parentTable    → links child tables to their parent
--       • design.populationMode → documents this is DDL-only
--
-- R7  Self-Documenting DDL
--     Every column carries an inline comment describing its
--     business meaning, source mapping, and (where applicable)
--     the semantic type transformation applied. Table-level
--     COMMENTs summarize purpose and cardinality.
--     Additional documentation patterns:
--       • Bronze-path breadcrumbs: every nested-data table has
--         a tree-style comment (└── notation) showing exactly
--         where the data was nested or sub-nested in the
--         bronze source (e.g. yelp_business → .attributes
--         → .Ambience)
--       • Layer labels: section headings include (Silver Cleaned)
--       • Example values: block comments above enriched tables
--         show a sample Python dict from the bronze source
--
-- R8  Consolidated Single-Query / Single-Schema Approach
--     ALL tables derived from the same bronze source table are
--     defined in ONE DDL query and created in ONE schema
--     (silver_cleaned). Both type-corrected and parsed-dict
--     tables coexist in the same schema. This keeps the full
--     transformation lineage visible in a single file and
--     avoids cross-schema FK dependencies.
--     The query is organized top-down: parent → level-1 children
--     → level-2 children → verification.
--
-- ════════════════════════════════════════════════════════════


-- ── 1. Parent Business Table (Silver Cleaned) ───────────────
-- Core identity, location, and summary metrics for each Yelp
-- business. One row per business. Serves as the anchor for all
-- child tables via business_id PK.

CREATE OR REPLACE TABLE yelp_poc_uc.silver_cleaned.yelp_business (
  business_id           STRING    NOT NULL  COMMENT 'Unique 22-char Yelp identifier for the business. Primary key.',
  name                  STRING              COMMENT 'Registered business name as displayed on Yelp.',
  address               STRING              COMMENT 'Street address of the business location.',
  city                  STRING              COMMENT 'City where the business is located.',
  state                 STRING              COMMENT 'Two-letter US state or Canadian province code (e.g. AZ, ON).',
  postal_code           STRING              COMMENT 'ZIP or postal code. Kept as STRING to preserve leading zeros.',
  latitude              DOUBLE              COMMENT 'Geographic latitude coordinate (WGS 84 datum).',
  longitude             DOUBLE              COMMENT 'Geographic longitude coordinate (WGS 84 datum).',
  stars                 DECIMAL(2,1)        COMMENT 'Average star rating from 1.0 to 5.0 in 0.5 increments. Source: DOUBLE cast to DECIMAL(2,1).',
  review_count          BIGINT              COMMENT 'Total number of user reviews received by this business.',
  is_open               BOOLEAN             COMMENT 'Whether the business is currently operating (true) or permanently closed (false). Source: BIGINT 0/1 cast to BOOLEAN.',
  CONSTRAINT pk_sc_business PRIMARY KEY (business_id)
)
USING DELTA
COMMENT 'A commercial establishment listed on the Yelp platform — its registered identity, physical location, aggregate star rating, review volume, and operational status. One row per business. Nested attributes, hours, and categories are normalized into child tables joined via FK on business_id.'
TBLPROPERTIES (
  'quality'               = 'silver_cleaned',
  'pipelines.sourceTable' = 'yelp_poc_uc.bronze_final.yelp_business',
  'design.populationMode' = 'ddl_only',
  'design.normalizedFrom' = 'yelp_poc_uc.bronze_final.yelp_business'
);


-- ── 2. Attributes Child Table (Silver Cleaned) ──────────────
-- Bronze path: yelp_business
--                └── .attributes  (STRUCT of 32 STRING fields)
-- Flattened business attributes originally stored as a STRUCT
-- of 32 STRING fields in the bronze source. Semantic types
-- are applied per column based on actual value inspection:
--   BOOLEAN (24) — source values are exclusively "True"/"False"
--   INT     (1)  — RestaurantsPriceRange2 values 1–4
--   STRING  (7)  — categorical enums with known domains
-- One row per business (1:1 with parent).

CREATE OR REPLACE TABLE yelp_poc_uc.silver_cleaned.yelp_business_attributes (
  business_id                              STRING   NOT NULL  COMMENT 'FK to yelp_business. Unique per row; also serves as PK for this table.',

  -- ─── Boolean attributes (source: STRING "True"/"False") ─────
  attr_accepts_insurance                   BOOLEAN   COMMENT 'Whether the business accepts insurance payments.',
  attr_bike_parking                        BOOLEAN   COMMENT 'Whether bike parking is available at the venue.',
  attr_business_accepts_bitcoin            BOOLEAN   COMMENT 'Whether the business accepts Bitcoin as payment.',
  attr_business_accepts_credit_cards       BOOLEAN   COMMENT 'Whether credit card payments are accepted.',
  attr_by_appointment_only                 BOOLEAN   COMMENT 'Whether the business operates by appointment only.',
  attr_byob                                BOOLEAN   COMMENT 'Whether customers can bring their own beverages (BYOB).',
  attr_caters                              BOOLEAN   COMMENT 'Whether the business offers catering services.',
  attr_coat_check                          BOOLEAN   COMMENT 'Whether a coat check service is provided.',
  attr_corkage                             BOOLEAN   COMMENT 'Whether customers can bring their own wine (corkage).',
  attr_dogs_allowed                        BOOLEAN   COMMENT 'Whether dogs or pets are allowed on premises.',
  attr_drive_thru                          BOOLEAN   COMMENT 'Whether a drive-through window is available.',
  attr_good_for_dancing                    BOOLEAN   COMMENT 'Whether the venue is suitable for dancing.',
  attr_good_for_kids                       BOOLEAN   COMMENT 'Whether the venue is family-friendly for children.',
  attr_happy_hour                          BOOLEAN   COMMENT 'Whether the venue offers happy hour promotions.',
  attr_has_tv                              BOOLEAN   COMMENT 'Whether TVs are available for viewing at the venue.',
  attr_open_24_hours                       BOOLEAN   COMMENT 'Whether the business operates around the clock.',
  attr_outdoor_seating                     BOOLEAN   COMMENT 'Whether outdoor or patio seating is available.',
  attr_restaurants_counter_service         BOOLEAN   COMMENT 'Whether the restaurant offers counter service.',
  attr_restaurants_delivery                BOOLEAN   COMMENT 'Whether the restaurant provides delivery service.',
  attr_restaurants_good_for_groups         BOOLEAN   COMMENT 'Whether the restaurant accommodates group dining.',
  attr_restaurants_reservations            BOOLEAN   COMMENT 'Whether the restaurant accepts advance reservations.',
  attr_restaurants_table_service           BOOLEAN   COMMENT 'Whether the restaurant provides seated table service.',
  attr_restaurants_take_out                BOOLEAN   COMMENT 'Whether the restaurant offers take-out orders.',
  attr_wheelchair_accessible               BOOLEAN   COMMENT 'Whether the venue is wheelchair accessible.',

  -- ─── Integer attribute (source: STRING "1"–"4") ─────────────
  attr_restaurants_price_range_2           INT       COMMENT 'Price tier: 1 = budget, 2 = moderate, 3 = pricey, 4 = upscale. Source: STRING cast to INT.',

  -- ─── Categorical string attributes (source: u''value'') ──────
  -- Cleaned values are lower_snake_case (DQ-BUSINESS-003). An explicit
  -- "no" is stored as the text false; null-like words (none, unknown,
  -- n/a, null) are stored as SQL NULL (DQ-BUSINESS-063).
  attr_ages_allowed                        STRING    COMMENT 'Age restriction policy (all_ages / 21_plus / 18_plus).',
  attr_alcohol                             STRING    COMMENT 'Alcohol service type (full_bar / beer_and_wine). Source value none is stored as NULL.',
  attr_byob_corkage                        STRING    COMMENT 'BYOB corkage policy (false / yes_free / yes_corkage). false = BYOB not allowed.',
  attr_noise_level                         STRING    COMMENT 'Typical ambient noise level (quiet / average / loud / very_loud).',
  attr_restaurants_attire                  STRING    COMMENT 'Expected dress code (casual / dressy / formal).',
  attr_smoking                             STRING    COMMENT 'Smoking policy (false / outdoor / yes). false = smoking not allowed.',
  attr_wifi                                STRING    COMMENT 'WiFi availability (free / false / paid). false = no WiFi.',


  CONSTRAINT pk_sc_business_attributes PRIMARY KEY (business_id),
  CONSTRAINT fk_sc_business_attributes_business
    FOREIGN KEY (business_id)
    REFERENCES yelp_poc_uc.silver_cleaned.yelp_business (business_id)
)
USING DELTA
COMMENT 'Operational and service characteristics of a Yelp business — policies (reservations, delivery, take-out), amenities (WiFi, TV, parking), pricing tier, ambience, and dietary offerings. 24 boolean flags, 1 integer price range, and 7 categorical enums. 1:1 with yelp_business.'
TBLPROPERTIES (
  'quality'               = 'silver_cleaned',
  'pipelines.sourceTable' = 'yelp_poc_uc.bronze_final.yelp_business',
  'design.populationMode' = 'ddl_only',
  'design.parentTable'    = 'yelp_poc_uc.silver_cleaned.yelp_business'
);


-- ── 3. Hours Child Table (Silver Cleaned) ───────────────────
-- Bronze path: yelp_business
--                └── .hours  (STRUCT of 7 STRING fields)
-- Weekly operating hours originally stored as a STRUCT of 7
-- STRING fields in the bronze source. Values are time-range
-- strings encoding an open-close pair (e.g. "8:0-22:0"), not
-- single timestamps, so STRING is the correct semantic type.
-- One row per business (1:1 with parent).

CREATE OR REPLACE TABLE yelp_poc_uc.silver_cleaned.yelp_business_hours (
  business_id                              STRING   NOT NULL  COMMENT 'FK to yelp_business. Unique per row; also serves as PK for this table.',
  hours_monday                             STRING             COMMENT 'Monday open-close time range (e.g. 8:0-22:0). NULL if closed.',
  hours_tuesday                            STRING             COMMENT 'Tuesday open-close time range. NULL if closed.',
  hours_wednesday                          STRING             COMMENT 'Wednesday open-close time range. NULL if closed.',
  hours_thursday                           STRING             COMMENT 'Thursday open-close time range. NULL if closed.',
  hours_friday                             STRING             COMMENT 'Friday open-close time range. NULL if closed.',
  hours_saturday                           STRING             COMMENT 'Saturday open-close time range. NULL if closed.',
  hours_sunday                             STRING             COMMENT 'Sunday open-close time range. NULL if closed.',
  CONSTRAINT pk_sc_business_hours PRIMARY KEY (business_id),
  CONSTRAINT fk_sc_business_hours_business
    FOREIGN KEY (business_id)
    REFERENCES yelp_poc_uc.silver_cleaned.yelp_business (business_id)
)
USING DELTA
COMMENT 'Weekly operating schedule of a Yelp business — open and close times for each day of the week. Values are time-range strings (e.g. 8:0-22:0); NULL means closed that day. 1:1 with yelp_business.'
TBLPROPERTIES (
  'quality'               = 'silver_cleaned',
  'pipelines.sourceTable' = 'yelp_poc_uc.bronze_final.yelp_business',
  'design.populationMode' = 'ddl_only',
  'design.parentTable'    = 'yelp_poc_uc.silver_cleaned.yelp_business'
);


-- ── 4. Categories Child Table (Silver Cleaned) ──────────────
-- Bronze path: yelp_business
--                └── .categories  (STRING, comma-separated list)
-- Business categories originally stored as a single comma-
-- separated STRING in the bronze source (e.g. "Restaurants,
-- Food, Bakeries"). Exploded into individual rows so each
-- category is queryable and indexable independently.
-- Zero-to-many rows per business (1:N with parent).

CREATE OR REPLACE TABLE yelp_poc_uc.silver_cleaned.yelp_business_categories (
  business_id                              STRING   NOT NULL  COMMENT 'FK to yelp_business. Part of the composite primary key.',
  category_name                            STRING   NOT NULL  COMMENT 'Individual business category label (e.g. Restaurants, Bakeries, Hair Salons).',
  category_position                        INT                COMMENT 'Ordinal position of this category in the original comma-separated list (0-based).',
  CONSTRAINT pk_sc_business_categories PRIMARY KEY (business_id, category_name),
  CONSTRAINT fk_sc_business_categories_business
    FOREIGN KEY (business_id)
    REFERENCES yelp_poc_uc.silver_cleaned.yelp_business (business_id)
)
USING DELTA
COMMENT 'Industry and service categories assigned to a Yelp business (e.g. Restaurants, Bakeries, Auto Repair). One row per category per business. Exploded from the bronze comma-separated categories string. 0:N with yelp_business.'
TBLPROPERTIES (
  'quality'               = 'silver_cleaned',
  'pipelines.sourceTable' = 'yelp_poc_uc.bronze_final.yelp_business',
  'design.populationMode' = 'ddl_only',
  'design.parentTable'    = 'yelp_poc_uc.silver_cleaned.yelp_business'
);


-- ════════════════════════════════════════════════════════════
-- PARSED JSON-DICT ATTRIBUTE TABLES (Silver Cleaned)
-- ════════════════════════════════════════════════════════════
-- The 7 JSON-dict STRING fields nested inside the bronze
-- attributes STRUCT are parsed directly into fully typed
-- BOOLEAN child tables. Each Python dict key becomes its own
-- column. All tables live in silver_cleaned.
-- ════════════════════════════════════════════════════════════


-- ── 5. Ambience (Silver Cleaned) ───────────────────────────
-- Bronze path: yelp_business
--                └── .attributes        (STRUCT)
--                      └── .Ambience    (STRING: Python dict, 9 boolean keys)
-- Describes the atmosphere and vibe of the venue.
-- Example: {'romantic': False, 'hipster': True, 'casual': True}
-- Cardinality: 0:1 with parent yelp_business (only businesses with the dict).

CREATE OR REPLACE TABLE yelp_poc_uc.silver_cleaned.yelp_business_ambience (
  business_id       STRING   NOT NULL  COMMENT 'FK to yelp_business. Unique per row; also serves as PK.',
  romantic          BOOLEAN            COMMENT 'Romantic or date-night atmosphere.',
  intimate          BOOLEAN            COMMENT 'Small, cozy, or intimate setting.',
  touristy          BOOLEAN            COMMENT 'Tourist-oriented or tourist-frequented venue.',
  hipster           BOOLEAN            COMMENT 'Hipster or trendy-alternative vibe.',
  divey             BOOLEAN            COMMENT 'Dive-bar or no-frills character.',
  classy            BOOLEAN            COMMENT 'Upscale-casual or sophisticated atmosphere.',
  trendy            BOOLEAN            COMMENT 'Currently fashionable or trendy venue.',
  upscale           BOOLEAN            COMMENT 'High-end or luxury atmosphere.',
  casual            BOOLEAN            COMMENT 'Relaxed, informal, everyday setting.',
  CONSTRAINT pk_sc_business_ambience PRIMARY KEY (business_id),
  CONSTRAINT fk_sc_business_ambience_business
    FOREIGN KEY (business_id)
    REFERENCES yelp_poc_uc.silver_cleaned.yelp_business (business_id)
)
USING DELTA
COMMENT 'Atmosphere and vibe of a Yelp business — whether the setting is romantic, casual, hipster, upscale, divey, etc. 9 boolean flags parsed from the attributes.Ambience dict. 0:1 with yelp_business (only businesses with the dict).'
TBLPROPERTIES (
  'quality'               = 'silver_cleaned',
  'pipelines.sourceTable' = 'yelp_poc_uc.bronze_final.yelp_business',
  'pipelines.sourceColumn'= 'attributes.Ambience',
  'design.populationMode' = 'ddl_only',
  'design.parentTable'    = 'yelp_poc_uc.silver_cleaned.yelp_business'
);


-- ── 6. Best Nights (Silver Cleaned) ────────────────────────
-- Bronze path: yelp_business
--                └── .attributes          (STRUCT)
--                      └── .BestNights    (STRING: Python dict, 7 day-of-week keys)
-- Flags which nights of the week are best to visit the venue.
-- Example: {'friday': True, 'saturday': True, 'monday': False}
-- Cardinality: 0:1 with parent yelp_business (only businesses with the dict).

CREATE OR REPLACE TABLE yelp_poc_uc.silver_cleaned.yelp_business_best_nights (
  business_id       STRING   NOT NULL  COMMENT 'FK to yelp_business. Unique per row; also serves as PK.',
  monday            BOOLEAN            COMMENT 'Monday is a recommended night to visit.',
  tuesday           BOOLEAN            COMMENT 'Tuesday is a recommended night to visit.',
  wednesday         BOOLEAN            COMMENT 'Wednesday is a recommended night to visit.',
  thursday          BOOLEAN            COMMENT 'Thursday is a recommended night to visit.',
  friday            BOOLEAN            COMMENT 'Friday is a recommended night to visit.',
  saturday          BOOLEAN            COMMENT 'Saturday is a recommended night to visit.',
  sunday            BOOLEAN            COMMENT 'Sunday is a recommended night to visit.',
  CONSTRAINT pk_sc_business_best_nights PRIMARY KEY (business_id),
  CONSTRAINT fk_sc_business_best_nights_business
    FOREIGN KEY (business_id)
    REFERENCES yelp_poc_uc.silver_cleaned.yelp_business (business_id)
)
USING DELTA
COMMENT 'Recommended nights to visit a Yelp business — which days of the week are considered the best experience. 7 day-of-week boolean flags. 0:1 with yelp_business (only businesses with the dict).'
TBLPROPERTIES (
  'quality'               = 'silver_cleaned',
  'pipelines.sourceTable' = 'yelp_poc_uc.bronze_final.yelp_business',
  'pipelines.sourceColumn'= 'attributes.BestNights',
  'design.populationMode' = 'ddl_only',
  'design.parentTable'    = 'yelp_poc_uc.silver_cleaned.yelp_business'
);


-- ── 7. Parking (Silver Cleaned) ────────────────────────────
-- Bronze path: yelp_business
--                └── .attributes              (STRUCT)
--                      └── .BusinessParking   (STRING: Python dict, 5 boolean keys)
-- Describes available parking options at or near the venue.
-- Example: {'garage': False, 'street': True, 'lot': True}
-- Cardinality: 0:1 with parent yelp_business (only businesses with the dict).

CREATE OR REPLACE TABLE yelp_poc_uc.silver_cleaned.yelp_business_parking (
  business_id       STRING   NOT NULL  COMMENT 'FK to yelp_business. Unique per row; also serves as PK.',
  garage            BOOLEAN            COMMENT 'Parking garage available nearby.',
  street            BOOLEAN            COMMENT 'Street parking available nearby.',
  validated         BOOLEAN            COMMENT 'Validated parking offered by the business.',
  lot               BOOLEAN            COMMENT 'Parking lot on or near premises.',
  valet             BOOLEAN            COMMENT 'Valet parking service offered.',
  CONSTRAINT pk_sc_business_parking PRIMARY KEY (business_id),
  CONSTRAINT fk_sc_business_parking_business
    FOREIGN KEY (business_id)
    REFERENCES yelp_poc_uc.silver_cleaned.yelp_business (business_id)
)
USING DELTA
COMMENT 'Parking options available at or near a Yelp business — garage, street, lot, valet, and validated parking. 5 boolean flags. 0:1 with yelp_business (only businesses with the dict).'
TBLPROPERTIES (
  'quality'               = 'silver_cleaned',
  'pipelines.sourceTable' = 'yelp_poc_uc.bronze_final.yelp_business',
  'pipelines.sourceColumn'= 'attributes.BusinessParking',
  'design.populationMode' = 'ddl_only',
  'design.parentTable'    = 'yelp_poc_uc.silver_cleaned.yelp_business'
);


-- ── 8. Dietary Restrictions (Silver Cleaned) ───────────────
-- Bronze path: yelp_business
--                └── .attributes                  (STRUCT)
--                      └── .DietaryRestrictions   (STRING: Python dict, 7 boolean keys)
-- Flags which dietary accommodations the business supports.
-- Example: {'vegan': True, 'gluten-free': True, 'halal': False}
-- Cardinality: 0:1 with parent yelp_business (only businesses with the dict).

CREATE OR REPLACE TABLE yelp_poc_uc.silver_cleaned.yelp_business_dietary_restrictions (
  business_id       STRING   NOT NULL  COMMENT 'FK to yelp_business. Unique per row; also serves as PK.',
  dairy_free        BOOLEAN            COMMENT 'Whether the business offers dairy-free menu options.',
  gluten_free       BOOLEAN            COMMENT 'Whether the business offers gluten-free menu options.',
  vegan             BOOLEAN            COMMENT 'Whether the business offers vegan menu options.',
  kosher            BOOLEAN            COMMENT 'Whether the business offers kosher-compliant options.',
  halal             BOOLEAN            COMMENT 'Whether the business offers halal-compliant options.',
  soy_free          BOOLEAN            COMMENT 'Whether the business offers soy-free menu options.',
  vegetarian        BOOLEAN            COMMENT 'Whether the business offers vegetarian menu options.',
  CONSTRAINT pk_sc_business_dietary_restrictions PRIMARY KEY (business_id),
  CONSTRAINT fk_sc_business_dietary_restrictions_business
    FOREIGN KEY (business_id)
    REFERENCES yelp_poc_uc.silver_cleaned.yelp_business (business_id)
)
USING DELTA
COMMENT 'Dietary accommodations offered by a Yelp business — vegan, vegetarian, gluten-free, halal, kosher, dairy-free, and soy-free options. 7 boolean flags. 0:1 with yelp_business (only businesses with the dict).'
TBLPROPERTIES (
  'quality'               = 'silver_cleaned',
  'pipelines.sourceTable' = 'yelp_poc_uc.bronze_final.yelp_business',
  'pipelines.sourceColumn'= 'attributes.DietaryRestrictions',
  'design.populationMode' = 'ddl_only',
  'design.parentTable'    = 'yelp_poc_uc.silver_cleaned.yelp_business'
);


-- ── 9. Good For Meal (Silver Cleaned) ──────────────────────
-- Bronze path: yelp_business
--                └── .attributes          (STRUCT)
--                      └── .GoodForMeal   (STRING: Python dict, 6 boolean keys)
-- Flags which meal occasions the restaurant is suited for.
-- Example: {'lunch': True, 'dinner': True, 'breakfast': False}
-- Cardinality: 0:1 with parent yelp_business (only businesses with the dict).

CREATE OR REPLACE TABLE yelp_poc_uc.silver_cleaned.yelp_business_good_for_meal (
  business_id       STRING   NOT NULL  COMMENT 'FK to yelp_business. Unique per row; also serves as PK.',
  breakfast         BOOLEAN            COMMENT 'Whether the venue is suitable for breakfast.',
  brunch            BOOLEAN            COMMENT 'Whether the venue is suitable for brunch.',
  lunch             BOOLEAN            COMMENT 'Whether the venue is suitable for lunch.',
  dinner            BOOLEAN            COMMENT 'Whether the venue is suitable for dinner.',
  latenight         BOOLEAN            COMMENT 'Whether the venue is suitable for late-night dining.',
  dessert           BOOLEAN            COMMENT 'Whether the venue is suitable for dessert or sweet treats.',
  CONSTRAINT pk_sc_business_good_for_meal PRIMARY KEY (business_id),
  CONSTRAINT fk_sc_business_good_for_meal_business
    FOREIGN KEY (business_id)
    REFERENCES yelp_poc_uc.silver_cleaned.yelp_business (business_id)
)
USING DELTA
COMMENT 'Meal occasions a Yelp business is suited for — breakfast, brunch, lunch, dinner, late-night, and dessert. 6 boolean flags. 0:1 with yelp_business (only businesses with the dict).'
TBLPROPERTIES (
  'quality'               = 'silver_cleaned',
  'pipelines.sourceTable' = 'yelp_poc_uc.bronze_final.yelp_business',
  'pipelines.sourceColumn'= 'attributes.GoodForMeal',
  'design.populationMode' = 'ddl_only',
  'design.parentTable'    = 'yelp_poc_uc.silver_cleaned.yelp_business'
);


-- ── 10. Hair Specializes In (Silver Cleaned) ───────────────
-- Bronze path: yelp_business
--                └── .attributes                (STRUCT)
--                      └── .HairSpecializesIn   (STRING: Python dict, 8 boolean keys)
-- Flags which hair-care specializations the salon offers.
-- Example: {'coloring': True, 'curly': True, 'kids': False}
-- Cardinality: 0:1 with parent yelp_business (only businesses with the dict).

CREATE OR REPLACE TABLE yelp_poc_uc.silver_cleaned.yelp_business_hair_specializes_in (
  business_id       STRING   NOT NULL  COMMENT 'FK to yelp_business. Unique per row; also serves as PK.',
  straightperms     BOOLEAN            COMMENT 'Whether the salon specializes in straightening perms.',
  coloring          BOOLEAN            COMMENT 'Whether the salon specializes in hair coloring or dyeing.',
  extensions        BOOLEAN            COMMENT 'Whether the salon specializes in hair extensions.',
  africanamerican   BOOLEAN            COMMENT 'Whether the salon specializes in African-American hair care.',
  curly             BOOLEAN            COMMENT 'Whether the salon specializes in curly hair styling.',
  kids              BOOLEAN            COMMENT 'Whether the salon specializes in children''s haircuts.',
  perms             BOOLEAN            COMMENT 'Whether the salon specializes in perms or permanent waves.',
  asian             BOOLEAN            COMMENT 'Whether the salon specializes in Asian hair care.',
  CONSTRAINT pk_sc_business_hair_specializes_in PRIMARY KEY (business_id),
  CONSTRAINT fk_sc_business_hair_specializes_in_business
    FOREIGN KEY (business_id)
    REFERENCES yelp_poc_uc.silver_cleaned.yelp_business (business_id)
)
USING DELTA
COMMENT 'Hair-care specializations offered by a Yelp salon — coloring, extensions, curly styling, perms, kids, and ethnic-hair expertise. 8 boolean flags. 0:1 with yelp_business (only businesses with the dict).'
TBLPROPERTIES (
  'quality'               = 'silver_cleaned',
  'pipelines.sourceTable' = 'yelp_poc_uc.bronze_final.yelp_business',
  'pipelines.sourceColumn'= 'attributes.HairSpecializesIn',
  'design.populationMode' = 'ddl_only',
  'design.parentTable'    = 'yelp_poc_uc.silver_cleaned.yelp_business'
);


-- ── 11. Music (Silver Cleaned) ─────────────────────────────
-- Bronze path: yelp_business
--                └── .attributes    (STRUCT)
--                      └── .Music   (STRING: Python dict, 7 boolean keys)
-- Flags which types of music or audio experience the venue offers.
-- Example: {'live': True, 'dj': False, 'background_music': True}
-- Cardinality: 0:1 with parent yelp_business (only businesses with the dict).

CREATE OR REPLACE TABLE yelp_poc_uc.silver_cleaned.yelp_business_music (
  business_id       STRING   NOT NULL  COMMENT 'FK to yelp_business. Unique per row; also serves as PK.',
  dj                BOOLEAN            COMMENT 'Whether a DJ performs or plays music at the venue.',
  background_music  BOOLEAN            COMMENT 'Whether background or ambient music is played.',
  no_music          BOOLEAN            COMMENT 'Whether the venue explicitly has no music.',
  jukebox           BOOLEAN            COMMENT 'Whether a jukebox is available for patron-selected music.',
  live              BOOLEAN            COMMENT 'Whether live music performances are hosted.',
  video             BOOLEAN            COMMENT 'Whether music videos or video screens are present.',
  karaoke           BOOLEAN            COMMENT 'Whether karaoke sessions are offered.',
  CONSTRAINT pk_sc_business_music PRIMARY KEY (business_id),
  CONSTRAINT fk_sc_business_music_business
    FOREIGN KEY (business_id)
    REFERENCES yelp_poc_uc.silver_cleaned.yelp_business (business_id)
)
USING DELTA
COMMENT 'Music and audio experience at a Yelp business — DJ, live performances, jukebox, karaoke, background music, or no music at all. 7 boolean flags. 0:1 with yelp_business (only businesses with the dict).'
TBLPROPERTIES (
  'quality'               = 'silver_cleaned',
  'pipelines.sourceTable' = 'yelp_poc_uc.bronze_final.yelp_business',
  'pipelines.sourceColumn'= 'attributes.Music',
  'design.populationMode' = 'ddl_only',
  'design.parentTable'    = 'yelp_poc_uc.silver_cleaned.yelp_business'
);


-- ── 12. Verification ────────────────────────────────────────

SELECT 'yelp_business' AS table_name, 'silver_cleaned' AS layer, COUNT(*) AS total_rows
FROM yelp_poc_uc.silver_cleaned.yelp_business
UNION ALL
SELECT 'yelp_business_attributes', 'silver_cleaned', COUNT(*)
FROM yelp_poc_uc.silver_cleaned.yelp_business_attributes
UNION ALL
SELECT 'yelp_business_hours', 'silver_cleaned', COUNT(*)
FROM yelp_poc_uc.silver_cleaned.yelp_business_hours
UNION ALL
SELECT 'yelp_business_categories', 'silver_cleaned', COUNT(*)
FROM yelp_poc_uc.silver_cleaned.yelp_business_categories
UNION ALL
SELECT 'yelp_business_ambience', 'silver_cleaned', COUNT(*)
FROM yelp_poc_uc.silver_cleaned.yelp_business_ambience
UNION ALL
SELECT 'yelp_business_best_nights', 'silver_cleaned', COUNT(*)
FROM yelp_poc_uc.silver_cleaned.yelp_business_best_nights
UNION ALL
SELECT 'yelp_business_parking', 'silver_cleaned', COUNT(*)
FROM yelp_poc_uc.silver_cleaned.yelp_business_parking
UNION ALL
SELECT 'yelp_business_dietary_restrictions', 'silver_cleaned', COUNT(*)
FROM yelp_poc_uc.silver_cleaned.yelp_business_dietary_restrictions
UNION ALL
SELECT 'yelp_business_good_for_meal', 'silver_cleaned', COUNT(*)
FROM yelp_poc_uc.silver_cleaned.yelp_business_good_for_meal
UNION ALL
SELECT 'yelp_business_hair_specializes_in', 'silver_cleaned', COUNT(*)
FROM yelp_poc_uc.silver_cleaned.yelp_business_hair_specializes_in
UNION ALL
SELECT 'yelp_business_music', 'silver_cleaned', COUNT(*)
FROM yelp_poc_uc.silver_cleaned.yelp_business_music;